In [1]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [2]:
from rag_helper import RAGBase
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [3]:
instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=instructions,
)

In [4]:
assistant.rag("How do I run Ollama locally?")

'To run Ollama locally:\n\n1. Install Ollama from: https://ollama.com/download  \n   - macOS: download and install the `.pkg`\n   - Windows: download and install the `.msi`\n   - Linux: run:\n     ```bash\n     curl -fsSL https://ollama.com/install.sh | sh\n     ```\n\n2. Open a terminal and start a model locally:\n   ```bash\n   ollama run llama3\n   ```\n   This downloads the LLaMA 3 model, starts it locally, and opens a chat-like interface.\n\n3. To test that the local server is running, use:\n   ```bash\n   curl http://localhost:11434\n   ```\n   You should get a response like:\n   ```json\n   {"models": [...]}  \n   ```\n\nIf you’re using Python, install the client with:\n\n```bash\npip install ollama\n```\n\nThen you can call it like this:\n\n```python\nimport ollama\n\nresponse = ollama.chat(\n    model=\'llama3\',\n    messages=[{"role": "user", "content": your_prompt}]\n)\n\nprint(response[\'message\'][\'content\'])\n```'

In [5]:
assistant.rag("How do I run Olama locally?")

'The FAQ doesn’t include instructions for running **Olama** locally.\n\nThe closest relevant guidance is that you can run the **course locally** instead of Codespaces if you’re comfortable setting up:\n\n- Python\n- `uv`\n- Jupyter\n- Docker\n- any other tools needed for the module\n\nIf you do run locally, make sure to **document your setup** and keep the environment **reproducible**.\n\nIf you meant something else by “Olama,” let me know.'

In [6]:
def rag(self, query):
    search_results = self.search(query)
    prompt = self.build_prompt(query, search_results)
    answer = self.llm(prompt)
    return answer

In [7]:
assistant.rag("How do I run Olama locally?")

'The FAQ doesn’t mention **Olama** specifically, but it does say you can run the course locally instead of using Codespaces.\n\nTo do that, you should be comfortable setting up:\n\n- Python\n- `uv`\n- Jupyter\n- Docker\n- any other tools needed for the module\n\nIf you run locally, make sure to:\n\n- document your setup\n- keep your environment reproducible\n\nIf you meant **Ollama**, I don’t have any course-specific instructions for running it locally in the FAQ context.'

In [30]:
messages = [
    {"role": "user", "content": "can i submit the project in atempt 2?"}
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
)

response.output_text

'Yes — if your course or platform allows multiple attempts, you can usually submit in Attempt 2.\n\nA couple of things to check:\n- Whether **Attempt 2 is currently unlocked**\n- Whether there’s a **deadline** for the second attempt\n- Whether your instructor/platform allows **resubmission after Attempt 1**\n\nIf you want, I can also help you phrase this more formally, like:\n> “Can I submit the project in Attempt 2?”\n\n'

In [32]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [33]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [34]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output

[ResponseFunctionToolCall(arguments='{"query":"submit project attempt 2 resubmission project submission attempt 2 FAQ"}', call_id='call_LuhfAWco38XXAJ2hbqpCQre9', name='search', type='function_call', id='fc_0460e47ead90d499006a627061feac81a0bcbb0751dab7d2ce', namespace=None, status='completed')]

In [35]:
import json

call = response.output[0]
args = json.loads(call.arguments)

results = search(**args)
result_json = json.dumps(results, indent=2)

In [36]:
messages.extend(response.output)

messages.append({
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json,
})

In [37]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output_text

'Yes — if your first submission doesn’t go through, you can improve it and resubmit in **attempt #2**.\n\nFrom the FAQ:\n- You only need to submit **one** project.\n- If the submission at **attempt #1** fails, you can resubmit during the **attempt #2** window.\n- If you missed attempt #1, you can still submit in attempt #2.\n\nIf you want, I can also explain what “attempt #1” and “attempt #2” mean for the capstone project.'

In [16]:
assistant.rag("How do I run Olama locally?")

'The FAQ context doesn’t mention **Olama** specifically.\n\nIf you meant running the course locally, the FAQ says you can do that instead of Codespaces if you’re comfortable setting up the needed tools locally: **Python, `uv`, Jupyter, Docker, and any other module-specific tools**. If you do, you should **document your setup and keep it reproducible**.\n\nIf you meant something else by “Olama,” let me know.'